# CircularIQ — Complete Google Colab Project

**An Intelligent Multi-Agent System for Institutional Circular Analysis and Action Tracking Using RAG**

This notebook builds the complete working prototype of CircularIQ using Python, Google Colab, Google Drive, Nexus API, ChromaDB, SQLite, PyMuPDF, LangGraph and ipywidgets.

## What the system does
- Uploads institutional circulars as PDFs.
- Extracts and chunks PDF text while preserving page numbers.
- Creates embeddings and stores searchable chunks in ChromaDB.
- Uses RAG to retrieve evidence for questions.
- Uses six specialized agents: Document Intelligence, Version Comparison, Conflict Detection, User Applicability, Deadline & Action, and Response & Evidence.
- Coordinates query-time agents using LangGraph.
- Stores document metadata in SQLite and files in Google Drive.

**API configuration:** Nexus chat model `nova-micro` and embedding model `text-embedding-3-small`. These model IDs were taken from the model list you supplied. The notebook tests the embeddings endpoint before ingestion; if your provider does not expose it in OpenAI-compatible format, the embedding setup cell will show the error and we can adjust it.

**Important limitations:** This is a student prototype, not a production college system. Scanned-PDF OCR, authentication, automatic email/calendar reminders and production-grade access controls are not included. The system must not treat a newer circular as authoritative unless the documents explicitly establish that.

## 1. Install packages
Run once. If Colab asks for a runtime restart, restart and continue.

In [ ]:
%pip -q install -U openai chromadb pymupdf langgraph ipywidgets pandas

## 2. Mount Google Drive and create folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/CircularIQ')
PDF_DIR = PROJECT_DIR / 'uploaded_pdfs'
TEXT_DIR = PROJECT_DIR / 'processed_text'
CHROMA_DIR = PROJECT_DIR / 'chroma_db'
DB_PATH = PROJECT_DIR / 'circulariq.sqlite3'

for folder in [PROJECT_DIR, PDF_DIR, TEXT_DIR, CHROMA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("CircularIQ folders created:", PROJECT_DIR)

## 3. Configure the Nexus API securely

We use Nexus for AI text generation. The API key is entered through a hidden prompt, so it is not printed in the notebook. Do not paste your API key into a code cell or share it in screenshots.

In [ ]:
from getpass import getpass
from openai import OpenAI

NEXUS_API_KEY = getpass("Enter your Nexus API key (hidden): ")

if not NEXUS_API_KEY:
    raise ValueError("Nexus API key is required.")

client = OpenAI(
    api_key=NEXUS_API_KEY,
    base_url="https://nexusapi.navigatelabs.ai"
)

GENERATION_MODEL = "nova-micro"
EMBEDDING_MODEL = "text-embedding-3-small"

print("Nexus API client configured.")
print("Generation model:", GENERATION_MODEL)
print("Embedding model:", EMBEDDING_MODEL)

## 3A. Test Nexus text generation
Run this small test before continuing. If it returns a short explanation, the chat model is reachable.

In [ ]:
test_response = client.chat.completions.create(
    model=GENERATION_MODEL,
    messages=[{"role": "user", "content": "Reply with: CircularIQ Nexus chat test successful."}]
)
print(test_response.choices[0].message.content)

## 4. Initialize ChromaDB and SQLite

In [ ]:
import re, json, sqlite3, hashlib
from datetime import datetime, timezone
from typing import TypedDict, Optional
import fitz
import chromadb
import pandas as pd
from langgraph.graph import StateGraph, END

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_or_create_collection(
    name="circulariq_chunks_nexus_v1",
    metadata={"hnsw:space": "cosine"}
)

def get_connection():
    return sqlite3.connect(DB_PATH)

with get_connection() as conn:
    conn.execute('''
        CREATE TABLE IF NOT EXISTS documents (
            document_id TEXT PRIMARY KEY,
            file_name TEXT NOT NULL,
            sha256 TEXT NOT NULL,
            uploaded_at TEXT NOT NULL,
            page_count INTEGER,
            summary TEXT,
            circular_type TEXT,
            effective_date TEXT
        )
    ''')
    conn.commit()

print("ChromaDB and SQLite are ready.")

## 5. Text cleaning, PDF extraction and chunking
Each chunk keeps its source page number so answers can cite the original PDF.

In [ ]:
def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def extract_pdf_pages(pdf_path):
    page_items = []
    with fitz.open(pdf_path) as pdf:
        for page_no, page in enumerate(pdf, start=1):
            text = clean_text(page.get_text("text"))
            if text:
                page_items.append({"page": page_no, "text": text})
    return page_items

def split_into_chunks(text, chunk_size=900, overlap=120):
    text = clean_text(text)
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            boundary = text.rfind(" ", start, end)
            if boundary > start + chunk_size // 2:
                end = boundary
        piece = text[start:end].strip()
        if piece:
            chunks.append(piece)
        if end >= len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks

print("PDF extraction and chunking functions ready.")

## 6. Create embeddings using Nexus

Embeddings convert text into numeric vectors. ChromaDB uses these vectors to find circular passages related to a question.

We use the Nexus model `text-embedding-3-small`. The same model must be used for both uploaded document chunks and user questions.

This notebook creates a new ChromaDB collection named `circulariq_chunks_nexus_v1` so it does not mix new embedding vectors with any older Gemini vectors.

In [ ]:
import time

def generate_embedding(text, task_type=None, max_retries=3):
    # task_type is accepted for compatibility with the existing project code.
    # OpenAI-compatible embedding APIs generally use input + model.
    for attempt in range(max_retries):
        try:
            response = client.embeddings.create(
                model=EMBEDDING_MODEL,
                input=text
            )
            return response.data[0].embedding
        except Exception as exc:
            if attempt == max_retries - 1:
                raise
            print(f"Nexus embedding request failed. Retry {attempt + 1}/{max_retries}: {exc}")
            time.sleep(3 * (2 ** attempt))

# Test the embedding endpoint before uploading PDFs.
test_vector = generate_embedding("Minimum attendance requirement is 75 percent.")
EMBEDDING_DIMENSION = len(test_vector)

print("Nexus embedding test passed.")
print("Embedding dimensions:", EMBEDDING_DIMENSION)

## 7. Document Intelligence Agent

This agent runs when a PDF is uploaded. It creates a short summary and identifies the circular type and effective date using Nexus.

In [ ]:
def call_nexus(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GENERATION_MODEL,
                messages=[
                    {"role": "system", "content": "You are a careful assistant for institutional circular analysis. Use only supplied document evidence and do not invent missing facts."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.2
            )
            answer = response.choices[0].message.content
            if not answer:
                raise RuntimeError("Nexus returned an empty response.")
            return answer.strip()
        except Exception as exc:
            if attempt == max_retries - 1:
                raise
            print(f"Nexus text generation temporarily unavailable. Retry {attempt + 1}/{max_retries}: {exc}")
            time.sleep(3 * (2 ** attempt))

# Compatibility alias: existing agent functions call call_gemini().
# They will now use Nexus without changing the six-agent workflow.
call_gemini = call_nexus

def document_intelligence_agent(file_name, full_text):
    prompt = f'''
You are CircularIQ's Document Intelligence Agent.
Analyze this institutional circular and return valid JSON only:
summary (2-4 sentences), circular_type, effective_date
(use "Not specified" if absent), main_topics (array).
Do not invent missing facts.

DOCUMENT: {file_name}
TEXT:
{full_text[:18000]}
'''
    raw = call_nexus(prompt)
    cleaned = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except Exception:
        return {
            "summary": raw,
            "circular_type": "Unclassified",
            "effective_date": "Not specified",
            "main_topics": []
        }

print("Nexus text-generation function and Document Intelligence Agent are ready.")

## 8. Upload and ingest PDF circulars
Select one or more PDFs. Originals, extracted text, embeddings and metadata are saved to Google Drive. Re-uploading an identical file is skipped.

In [ ]:
from google.colab import files
uploaded_files = files.upload()

def ingest_pdf(file_name, file_bytes):
    if not file_name.lower().endswith(".pdf"):
        print("Skipped non-PDF:", file_name)
        return

    file_hash = hashlib.sha256(file_bytes).hexdigest()
    document_id = hashlib.md5((file_name + file_hash).encode()).hexdigest()
    pdf_path = PDF_DIR / file_name
    pdf_path.write_bytes(file_bytes)

    # Re-ingestion is safe: chunk IDs are deterministic and ChromaDB upsert updates them.
    # This also allows PDFs from an earlier Gemini-based run to be added to the new Nexus collection.
    pages = extract_pdf_pages(pdf_path)
    if not pages:
        print("No selectable text found. Scanned PDF OCR is not included in this version:", file_name)
        return

    full_text = "\n\n".join(p["text"] for p in pages)
    (TEXT_DIR / f"{document_id}.txt").write_text(full_text, encoding="utf-8")

    records = []
    for page in pages:
        for chunk_no, chunk in enumerate(split_into_chunks(page["text"])):
            records.append({"text": chunk, "page": page["page"], "chunk": chunk_no})

    if not records:
        print("No text chunks:", file_name)
        return

    vectors = [
        generate_embedding(r["text"], "RETRIEVAL_DOCUMENT")
        for r in records
    ]
    ids = [f"{document_id}_chunk_{i}" for i in range(len(records))]
    metadata = [{
        "document_id": document_id,
        "document_name": file_name,
        "page": int(r["page"]),
        "chunk_number": int(r["chunk"])
    } for r in records]

    collection.upsert(
        ids=ids,
        documents=[r["text"] for r in records],
        embeddings=vectors,
        metadatas=metadata
    )

    intel = document_intelligence_agent(file_name, full_text)
    with get_connection() as conn:
        conn.execute('''
            INSERT OR REPLACE INTO documents
            (document_id, file_name, sha256, uploaded_at, page_count, summary, circular_type, effective_date)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            document_id, file_name, file_hash,
            datetime.now(timezone.utc).isoformat(), len(pages),
            intel.get("summary", ""),
            intel.get("circular_type", "Unclassified"),
            intel.get("effective_date", "Not specified")
        ))
        conn.commit()

    print(f"Ingested {file_name}: {len(pages)} pages, {len(records)} chunks")
    print("Summary:", intel.get("summary", ""))

for filename, file_bytes in uploaded_files.items():
    try:
        ingest_pdf(filename, file_bytes)
    except Exception as exc:
        print(f"Could not ingest {filename}: {type(exc).__name__}: {exc}")

## 9. View uploaded document records

In [ ]:
with get_connection() as conn:
    documents_df = pd.read_sql_query(
        "SELECT file_name, uploaded_at, page_count, circular_type, effective_date, summary FROM documents ORDER BY uploaded_at DESC",
        conn
    )
display(documents_df)
print("Chunks in ChromaDB:", collection.count())

## 10. RAG retrieval
The question is embedded with `RETRIEVAL_QUERY` and matched against stored document chunks.

In [ ]:
def retrieve_context(question, top_k=5):
    total = collection.count()
    if total == 0:
        return []
    q_vector = generate_embedding(question)
    result = collection.query(
        query_embeddings=[q_vector],
        n_results=min(top_k, total),
        include=["documents", "metadatas", "distances"]
    )
    hits = []
    for i, text in enumerate(result["documents"][0]):
        hits.append({
            "text": text,
            "metadata": result["metadatas"][0][i],
            "distance": result["distances"][0][i]
        })
    return hits

def format_sources(hits):
    return "\n\n".join(
        f"[Source: {h['metadata'].get('document_name')}, page {h['metadata'].get('page')}]\\n{h['text']}"
        for h in hits
    )

print("Retriever ready. Upload at least one PDF before asking questions.")

## 11. Six CircularIQ agents

1. **Document Intelligence Agent** — analyzes each uploaded circular during ingestion.
2. **Version Comparison Agent** — compares older and newer circular evidence when asked.
3. **Conflict Detection Agent** — identifies contradictory or materially different instructions.
4. **User Applicability Agent** — checks explicit rules against a supplied user profile.
5. **Deadline & Action Agent** — extracts deadlines, dates, tasks and responsible parties.
6. **Response & Evidence Agent** — produces the final grounded answer with filename and page citations.

The five query-time agents call the shared `call_nexus()` function. A compatibility alias named `call_gemini` is retained in the code so the original agent functions continue to work without rewriting every function.

In [ ]:
def version_comparison_agent(question, hits):
    triggers = ["version", "revised", "revision", "changed", "change", "previous", "compare", "difference", "updated"]
    if not any(w in question.lower() for w in triggers):
        return "Not triggered."
    prompt = f'''
You are the Version Comparison Agent. Compare only the retrieved evidence.
Identify additions, removals and modifications. Cite filenames/pages.
Do not say one circular supersedes another unless evidence explicitly says so.
If evidence is insufficient, say so.
QUESTION: {question}
EVIDENCE:
{format_sources(hits)}
'''
    return call_gemini(prompt)

def conflict_detection_agent(question, hits):
    if len(hits) < 2:
        return "Not enough retrieved evidence to assess conflicts."
    prompt = f'''
You are the Conflict Detection Agent. Identify instructions that conflict or materially differ.
Distinguish a true contradiction from an update or additional detail.
Cite filenames/pages. Do not decide which rule wins unless evidence explicitly establishes precedence.
If no conflict is found, say so.
QUESTION: {question}
EVIDENCE:
{format_sources(hits)}
'''
    return call_gemini(prompt)

def applicability_agent(question, hits, user_profile):
    if not user_profile:
        return "Not run: no user profile supplied."
    prompt = f'''
You are the User Applicability Agent. Determine whether the retrieved rules apply to this profile.
Use only explicit audience and eligibility conditions. If details are missing, say "Cannot determine".
Do not infer personal details.
PROFILE: {json.dumps(user_profile)}
QUESTION: {question}
EVIDENCE:
{format_sources(hits)}
'''
    return call_gemini(prompt)

def deadline_action_agent(question, hits):
    prompt = f'''
You are the Deadline & Action Agent. Extract only relevant dates, deadlines, times,
required actions, responsible persons and eligibility conditions explicitly stated in evidence.
If none are present, say so. Do not invent dates or calculate deadlines.
Cite filenames/pages.
QUESTION: {question}
EVIDENCE:
{format_sources(hits)}
'''
    return call_gemini(prompt)

def response_evidence_agent(question, hits, agent_notes):
    if not hits:
        return "I could not find relevant information in the uploaded circulars. Please upload a relevant circular or clarify the question."
    prompt = f'''
You are CircularIQ's Response & Evidence Agent.
Answer using ONLY the evidence below.
Do not invent rules, dates, deadlines, eligibility or policy precedence.
If evidence is insufficient, clearly say so.
Cite each factual claim as [filename, page N].
If documents differ, explain the difference; do not assume the newer one overrides the older one.
Use simple English.

QUESTION: {question}
RETRIEVED EVIDENCE:
{format_sources(hits)}
OTHER AGENT NOTES:
{json.dumps(agent_notes, ensure_ascii=False)}
ANSWER:
'''
    return call_gemini(prompt)

print("Six agent functions are ready.")

## 12. Coordinate agents with LangGraph
The graph runs the specialized agents in sequence. Agents can return 'not triggered' when their task is not relevant.

In [ ]:
class CircularState(TypedDict, total=False):
    question: str
    user_profile: dict
    hits: list
    version_analysis: str
    conflict_analysis: str
    applicability_analysis: str
    deadline_analysis: str
    answer: str

def version_node(state):
    return {"version_analysis": version_comparison_agent(state["question"], state["hits"])}

def conflict_node(state):
    q = state["question"].lower()
    triggers = ["conflict", "contradiction", "contradict", "different", "compare", "version", "revised", "changed"]
    if not any(w in q for w in triggers):
        return {"conflict_analysis": "Not triggered."}
    return {"conflict_analysis": conflict_detection_agent(state["question"], state["hits"])}

def applicability_node(state):
    return {"applicability_analysis": applicability_agent(
        state["question"], state["hits"], state.get("user_profile", {})
    )}

def deadline_node(state):
    q = state["question"].lower()
    triggers = ["deadline", "due", "when", "date", "submit", "apply", "action", "last date", "reminder"]
    if not any(w in q for w in triggers):
        return {"deadline_analysis": "Not triggered."}
    return {"deadline_analysis": deadline_action_agent(state["question"], state["hits"])}

def answer_node(state):
    notes = {
        "version_comparison": state.get("version_analysis", ""),
        "conflict_detection": state.get("conflict_analysis", ""),
        "applicability": state.get("applicability_analysis", ""),
        "deadline_action": state.get("deadline_analysis", "")
    }
    return {"answer": response_evidence_agent(state["question"], state["hits"], notes)}

builder = StateGraph(CircularState)
builder.add_node("version_comparison", version_node)
builder.add_node("conflict_detection", conflict_node)
builder.add_node("user_applicability", applicability_node)
builder.add_node("deadline_action", deadline_node)
builder.add_node("response_evidence", answer_node)
builder.set_entry_point("version_comparison")
builder.add_edge("version_comparison", "conflict_detection")
builder.add_edge("conflict_detection", "user_applicability")
builder.add_edge("user_applicability", "deadline_action")
builder.add_edge("deadline_action", "response_evidence")
builder.add_edge("response_evidence", END)

circulariq_graph = builder.compile()
print("LangGraph workflow compiled.")

## 13. Main CircularIQ function

In [ ]:
def ask_circulariq(question, user_profile=None, top_k=5, show_agent_steps=True):
    question = question.strip()
    if not question:
        return "Please enter a question."

    hits = retrieve_context(question, top_k)
    if not hits:
        return "No relevant circular text was retrieved. Upload PDF circulars first."

    state = {
        "question": question,
        "user_profile": user_profile or {},
        "hits": hits
    }
    result = circulariq_graph.invoke(state)

    if show_agent_steps:
        print("VERSION COMPARISON AGENT:\\n", result.get("version_analysis", ""))
        print("\\nCONFLICT DETECTION AGENT:\\n", result.get("conflict_analysis", ""))
        print("\\nUSER APPLICABILITY AGENT:\\n", result.get("applicability_analysis", ""))
        print("\\nDEADLINE & ACTION AGENT:\\n", result.get("deadline_analysis", ""))
        print("\\nRETRIEVED SOURCES:")
        for hit in hits:
            m = hit["metadata"]
            print("-", m.get("document_name"), "page", m.get("page"), "distance", round(hit["distance"], 4))
        print("\\nFINAL ANSWER:\\n")
    return result.get("answer", "No answer generated.")

print("CircularIQ is ready.")

## 14. Test: basic RAG question

In [ ]:
answer = ask_circulariq(
    "What is the minimum attendance percentage required?",
    show_agent_steps=True
)
print(answer)

## 15. Test: user applicability
Use a fictional profile for testing. Do not enter sensitive personal information.

In [ ]:
test_profile = {
    "role": "student",
    "department": "Computer Science",
    "year": "Third year",
    "programme": "BE"
}
answer = ask_circulariq(
    "Which attendance rules apply to this student?",
    user_profile=test_profile,
    show_agent_steps=True
)
print(answer)

## 16. Test: deadlines and actions

In [ ]:
answer = ask_circulariq(
    "What are the upcoming deadlines and what actions should students take?",
    user_profile={"role": "student"},
    show_agent_steps=True
)
print(answer)

## 17. Test: compare policy versions
Upload both versions of the same circular before running this test.

In [ ]:
answer = ask_circulariq(
    "Compare the revised attendance policy with the previous version. What changed, and are there any conflicts?",
    user_profile={"role": "student"},
    show_agent_steps=True
)
print(answer)

## 18. Simple interactive notebook interface

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

question_box = widgets.Textarea(
    placeholder="Ask a question about uploaded circulars...",
    description="Question:",
    layout=widgets.Layout(width="100%", height="90px")
)
role_box = widgets.Dropdown(
    options=["Not provided", "Student", "Faculty", "Class Advisor", "Administrator"],
    value="Not provided",
    description="Role:"
)
ask_button = widgets.Button(description="Ask CircularIQ", button_style="primary")
output_area = widgets.Output()

def on_ask_clicked(_):
    with output_area:
        clear_output()
        profile = {} if role_box.value == "Not provided" else {"role": role_box.value}
        try:
            answer = ask_circulariq(question_box.value, user_profile=profile, show_agent_steps=False)
            display(Markdown("### CircularIQ Answer\\n" + answer))
        except Exception as exc:
            print(f"{type(exc).__name__}: {exc}")
            print("If Nexus returns a temporary service error, wait and retry.")

ask_button.on_click(on_ask_clicked)
display(question_box, role_box, ask_button, output_area)

## 19. View document records and storage status

In [ ]:
with get_connection() as conn:
    final_docs = pd.read_sql_query(
        "SELECT file_name, uploaded_at, page_count, circular_type, effective_date, summary FROM documents ORDER BY uploaded_at DESC",
        conn
    )
display(final_docs)
print("ChromaDB chunk count:", collection.count())
print("Google Drive project folder:", PROJECT_DIR)

# How the complete project works

```text
PDF upload
   |
PyMuPDF extraction
   |
Text cleaning and chunking (page metadata retained)
   |
Nexus document embeddings
   |
ChromaDB vector storage + SQLite document metadata
   |
Document Intelligence Agent (summary and metadata)

User question
   |
Nexus query embedding
   |
ChromaDB semantic retrieval
   |
LangGraph agent coordination
   |-- Version Comparison Agent
   |-- Conflict Detection Agent
   |-- User Applicability Agent
   |-- Deadline & Action Agent
   |-- Response & Evidence Agent
   |
Grounded answer with filename and page citations
```

## Limitations and next improvements
- **Scanned PDFs:** OCR is not included; this version needs selectable text.
- **Version governance:** the version agent compares retrieved evidence. A production version should add explicit policy IDs, version numbers, effective dates and supersession links in SQLite.
- **Conflict handling:** conflicts are flagged for human review; the system does not decide which policy is legally authoritative.
- **Reminders:** deadlines are extracted, but email/calendar notifications are not implemented.
- **Production readiness:** add authentication, role-based access, audit logs, file validation and retention controls before using real institutional data.
- **Nexus:** the Navigate Lab key cannot be integrated until its base URL, model name and API format are provided. Do not substitute it for the Nexus key.

Official references:
- Google Gen AI SDK: https://googleapis.github.io/python-genai/
- Colab file upload and Drive I/O: https://colab.research.google.com/notebooks/io.ipynb